### Load the master dataset

In [32]:
import pandas as pd

master = pd.read_csv(
    "../data/processed/master_dataset.csv",
    parse_dates=["utc_timestamp", "time"]
)
master.info()

<class 'pandas.DataFrame'>
RangeIndex: 50400 entries, 0 to 50399
Data columns (total 13 columns):
 #   Column                              Non-Null Count  Dtype              
---  ------                              --------------  -----              
 0   utc_timestamp                       50400 non-null  datetime64[us, UTC]
 1   DE_load_actual_entsoe_transparency  50400 non-null  float64            
 2   DE_solar_generation_actual          50297 non-null  float64            
 3   DE_wind_generation_actual           50326 non-null  float64            
 4   DE_solar_capacity                   43799 non-null  float64            
 5   DE_wind_capacity                    43799 non-null  float64            
 6   DE_LU_price_day_ahead               17540 non-null  float64            
 7   time                                50400 non-null  datetime64[us, UTC]
 8   temperature_C                       50400 non-null  float64            
 9   humidity_pct                        50400 non-null

In [33]:
import pandas as pd

master = pd.read_csv(
    "../data/processed/master_dataset.csv",
    parse_dates=["utc_timestamp", "time"]
)

master.info()

<class 'pandas.DataFrame'>
RangeIndex: 50400 entries, 0 to 50399
Data columns (total 13 columns):
 #   Column                              Non-Null Count  Dtype              
---  ------                              --------------  -----              
 0   utc_timestamp                       50400 non-null  datetime64[us, UTC]
 1   DE_load_actual_entsoe_transparency  50400 non-null  float64            
 2   DE_solar_generation_actual          50297 non-null  float64            
 3   DE_wind_generation_actual           50326 non-null  float64            
 4   DE_solar_capacity                   43799 non-null  float64            
 5   DE_wind_capacity                    43799 non-null  float64            
 6   DE_LU_price_day_ahead               17540 non-null  float64            
 7   time                                50400 non-null  datetime64[us, UTC]
 8   temperature_C                       50400 non-null  float64            
 9   humidity_pct                        50400 non-null

### Create a working copy of the master dataset

In [34]:
master_clean = master.copy()

print(master_clean.shape)

(50400, 13)


### cleaning the solar generation missing values

#### Using interpolation

In [35]:
master_clean[
    'DE_solar_generation_actual'
] = master_clean[
    'DE_solar_generation_actual'
].interpolate()

master_clean[
    'DE_solar_generation_actual'
].isnull().sum()

np.int64(7)

#### Using backward fill to fill the remaining missing values

In [36]:
master_clean[
    'DE_solar_generation_actual'
] = master_clean[
    'DE_solar_generation_actual'
].bfill()

master_clean[
    'DE_solar_generation_actual'
].isnull().sum()

np.int64(0)

### Using interpolation and backward fill to handle missing values in Wind generation data

In [37]:
master_clean[
    'DE_wind_generation_actual'
] = master_clean[
    'DE_wind_generation_actual'
].interpolate()

master_clean[
    'DE_wind_generation_actual'
] = master_clean[
    'DE_wind_generation_actual'
].bfill()

master_clean[
    'DE_wind_generation_actual'
].isnull().sum()

np.int64(0)

### Handling missing values in solar and wind capacity column using forward fill

In [38]:
master_clean['DE_solar_capacity'] = (
    master_clean['DE_solar_capacity']
    .ffill()
)

master_clean['DE_wind_capacity'] = (
    master_clean['DE_wind_capacity']
    .ffill()
)

master_clean[
    [
        'DE_solar_capacity',
        'DE_wind_capacity'
    ]
].isnull().sum()

DE_solar_capacity    0
DE_wind_capacity     0
dtype: int64

### Verifying overall missing values after handling some missing values in previous workflow

In [39]:
master_clean.isnull().sum()

utc_timestamp                             0
DE_load_actual_entsoe_transparency        0
DE_solar_generation_actual                0
DE_wind_generation_actual                 0
DE_solar_capacity                         0
DE_wind_capacity                          0
DE_LU_price_day_ahead                 32860
time                                      0
temperature_C                             0
humidity_pct                              0
cloud_cover_pct                           0
wind_speed_ms                             0
precipitation_mm                          0
dtype: int64

*We will leave the price column untouched for now because we already established that the missingness is due to historical availability, not bad data.*

### creating the first and most important time feature.

In [40]:
master_clean['hour'] = (
    master_clean['utc_timestamp']
    .dt.hour
)

master_clean[
    ['utc_timestamp', 'hour']
].head()

,utc_timestamp,hour
0,2015-01-01 00:00:00+00:00,0
1,2015-01-01 01:00:00+00:00,1
2,2015-01-01 02:00:00+00:00,2
3,2015-01-01 03:00:00+00:00,3
4,2015-01-01 04:00:00+00:00,4


### Creating the day of week feature

In [41]:
master_clean['day_of_week'] = (
    master_clean['utc_timestamp']
    .dt.dayofweek
)

master_clean[
    ['utc_timestamp', 'day_of_week']
].head()

,utc_timestamp,day_of_week
0,2015-01-01 00:00:00+00:00,3
1,2015-01-01 01:00:00+00:00,3
2,2015-01-01 02:00:00+00:00,3
3,2015-01-01 03:00:00+00:00,3
4,2015-01-01 04:00:00+00:00,3


### Interpretation:

0 = Monday  
1 = Tuesday  
2 = Wednesday  
3 = Thursday  
4 = Friday  
5 = Saturday  
6 = Sunday

### Creating the month feature

In [42]:
master_clean['month'] = (
    master_clean['utc_timestamp']
    .dt.month
)

master_clean[
    ['utc_timestamp', 'month']
].head()

,utc_timestamp,month
0,2015-01-01 00:00:00+00:00,1
1,2015-01-01 01:00:00+00:00,1
2,2015-01-01 02:00:00+00:00,1
3,2015-01-01 03:00:00+00:00,1
4,2015-01-01 04:00:00+00:00,1


### Creating the year feature

In [43]:
master_clean['year'] = (
    master_clean['utc_timestamp']
    .dt.year
)

master_clean[
    ['utc_timestamp', 'year']
].head()

,utc_timestamp,year
0,2015-01-01 00:00:00+00:00,2015
1,2015-01-01 01:00:00+00:00,2015
2,2015-01-01 02:00:00+00:00,2015
3,2015-01-01 03:00:00+00:00,2015
4,2015-01-01 04:00:00+00:00,2015


### Creating the remaining calendar features

In [44]:
master_clean['is_weekend'] = (
    master_clean['day_of_week'] >= 5
).astype(int)

master_clean['quarter'] = (
    master_clean['utc_timestamp']
    .dt.quarter
)

master_clean['week_of_year'] = (
    master_clean['utc_timestamp']
    .dt.isocalendar()
    .week
)

master_clean['season'] = (
    master_clean['month']
    .map({
        12:'Winter',
        1:'Winter',
        2:'Winter',
        3:'Spring',
        4:'Spring',
        5:'Spring',
        6:'Summer',
        7:'Summer',
        8:'Summer',
        9:'Autumn',
        10:'Autumn',
        11:'Autumn'
    })
)

master_clean[
    [
        'utc_timestamp',
        'day_of_week',
        'is_weekend',
        'month',
        'quarter',
        'week_of_year',
        'season'
    ]
].head()

,utc_timestamp,day_of_week,is_weekend,month,quarter,week_of_year,season
0,2015-01-01 00:00:00+00:00,3,0,1,1,1,Winter
1,2015-01-01 01:00:00+00:00,3,0,1,1,1,Winter
2,2015-01-01 02:00:00+00:00,3,0,1,1,1,Winter
3,2015-01-01 03:00:00+00:00,3,0,1,1,1,Winter
4,2015-01-01 04:00:00+00:00,3,0,1,1,1,Winter


### Creating a master cleaned csv file for EDA 

In [45]:
master_clean.to_csv(
    "../data/processed/master_clean.csv",
    index=False
)